# Item-Item Collaborative Filtering Recommendation System
## Using Co-Occurrence and Reader Similarity

This notebook implements a collaborative filtering system that:
- Analyzes which books are read by the same users
- Builds a book-to-book co-occurrence matrix
- Computes similarity based on shared readers
- Provides recommendations based on books similar users have read
- **Testable on ANY user in the dataset**

### System Architecture
1. **Data Loading**: Parse user-book relationships from CSV
2. **Co-Occurrence Matrix**: Count shared readers between books
3. **Similarity Computation**: Jaccard or Cosine similarity
4. **Recommendation Engine**: Find similar books to what user previously read
5. **Hybrid Option**: Combine with content-based recommendations

## Section 1: Load and Parse the Book Data

In [55]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

# Load the user-book data
user_books_df = pd.read_csv('company_u.csv', header=0)
user_books_df.columns = ['User', 'Books']

print("Loaded user data:")
print(f"Total users: {len(user_books_df)}")
print(f"\nFirst 3 users:")
print(user_books_df.head(3))
print(f"\nUser IDs range: {user_books_df['User'].min()} to {user_books_df['User'].max()}")

Loaded user data:
Total users: 298

First 3 users:
   User                       Books
0 User002 A guide to the project management body of know...
1 User003    Ideology : an introduction / Terry Eagleton.
2 User004 How to think on your feet : a revolutionary te...

User IDs range: User002 to User300


In [24]:
# Parse books: split by " / " (new book delimiter)
def parse_books(books_string):
  """
  Parse books from a user's book string.
  Books are delimited by " / "
  Extract title (before " / by " or " ; ")
  """
  if pd.isna(books_string):
    return []
  
  books_str = str(books_string)
  books = books_str.split(' / ')
  
  titles = []
  for book in books:
    # Remove the author part (after " by " or " ; ")
    if ' by ' in book:
      title = book.split(' by ')[0].strip()
    elif ' ; ' in book:
      title = book.split(' ; ')[0].strip()
    else:
      title = book.strip()
    
    title = title.rstrip('.')
    
    if title:
      titles.append(title)
  
  return titles

# Apply parsing to create user-book pairs
user_book_pairs = []
for _, row in user_books_df.iterrows():
  user = row['User']
  books = parse_books(row['Books'])
  for book in books:
    user_book_pairs.append({'User': user, 'Title': book})

user_books_parsed = pd.DataFrame(user_book_pairs)
print(f"Total user-book pairs: {len(user_books_parsed)}")
print(f"Unique books: {user_books_parsed['Title'].nunique()}")
print(f"Unique users: {user_books_parsed['User'].nunique()}")

print(f"\nFirst 5 parsed pairs:")
print(user_books_parsed.head())

Total user-book pairs: 999
Unique books: 909
Unique users: 298

First 5 parsed pairs:
   User                       Title
0 User002 A guide to the project management body of know...
1 User002            Project Management Institute
2 User003             Ideology : an introduction
3 User003                   Terry Eagleton
4 User004 How to think on your feet : a revolutionary te...


## Section 2: Build User-Book Matrix and Co-Occurrence Matrix

In [56]:
# Create unique list of books and users
unique_books = sorted(user_books_parsed['Title'].unique())
unique_users = sorted(user_books_parsed['User'].unique())

print(f"Total unique books: {len(unique_books)}")
print(f"Total unique users: {len(unique_users)}")

# Create mapping dictionaries
book_to_idx = {book: idx for idx, book in enumerate(unique_books)}
user_to_idx = {user: idx for idx, user in enumerate(unique_users)}

# Create user-book matrix (sparse)
rows = [user_to_idx[row['User']] for _, row in user_books_parsed.iterrows()]
cols = [book_to_idx[row['Title']] for _, row in user_books_parsed.iterrows()]
data = [1] * len(user_books_parsed)

user_book_matrix = csr_matrix((data, (rows, cols)), shape=(len(unique_users), len(unique_books)))

print(f"\nUser-Book Matrix Shape: {user_book_matrix.shape}")
print(f"Sparsity: {(1 - user_book_matrix.nnz / (user_book_matrix.shape[0] * user_book_matrix.shape[1])) * 100:.1f}%")
print(f"Total interactions: {user_book_matrix.nnz}")

Total unique books: 909
Total unique users: 298

User-Book Matrix Shape: (298, 909)
Sparsity: 99.6%
Total interactions: 971


In [26]:
# Build book-to-book co-occurrence matrix
# This measures how many users read BOTH books

print("Building co-occurrence matrix...")
# Multiply U_T * U to get book-book co-occurrence
co_occurrence_matrix = user_book_matrix.T @ user_book_matrix

# Convert to dense for easier computation
co_occurrence_dense = co_occurrence_matrix.toarray()

print(f"\nCo-Occurrence Matrix Shape: {co_occurrence_dense.shape}")
print(f" - Rows: books")
print(f" - Cols: books")
print(f" - Values: number of shared readers")

# Display co-occurrence statistics
co_occurrence_values = co_occurrence_dense[np.triu_indices_from(co_occurrence_dense, k=1)]
print(f"\nCo-Occurrence Statistics:")
print(f" Mean shared readers: {co_occurrence_values.mean():.2f}")
print(f" Max shared readers: {co_occurrence_values.max():.0f}")
print(f" Min shared readers: {co_occurrence_values.min():.0f}")
print(f" Books with >5 shared readers: {(co_occurrence_values > 5).sum()}")
print(f" Books with >10 shared readers: {(co_occurrence_values > 10).sum()}")

Building co-occurrence matrix...

Co-Occurrence Matrix Shape: (909, 909)
 - Rows: books
 - Cols: books
 - Values: number of shared readers

Co-Occurrence Statistics:
 Mean shared readers: 0.00
 Max shared readers: 4
 Min shared readers: 0
 Books with >5 shared readers: 0
 Books with >10 shared readers: 0


## Section 3: Compute Book Similarity

In [57]:
# Normalize co-occurrence matrix to get similarity scores
# We'll use multiple similarity metrics

def jaccard_similarity(co_occur_matrix):
  """
  Compute Jaccard similarity from co-occurrence matrix
  J(A,B) = |A ∩ B| / |A ∪ B|
  """
  # Get number of readers for each book
  book_counts = np.array(co_occur_matrix.diagonal())
  
  # Compute Jaccard similarity
  jaccard = co_occur_matrix / (book_counts[:, None] + book_counts[None, :] - co_occur_matrix + 1e-8)
  
  return jaccard

# Compute both similarity metrics
jaccard_sim = jaccard_similarity(co_occurrence_dense)

# Also compute cosine similarity on the user-book matrix directly
# This is another common CF approach
cosine_sim = cosine_similarity(user_book_matrix.T)

# We'll use a weighted combination
# 60% Jaccard (focused on shared readers) + 40% Cosine (broader similarity)
book_similarity_matrix = 0.6 * jaccard_sim + 0.4 * cosine_sim

# Set diagonal to 0 (book not similar to itself for recommendations)
np.fill_diagonal(book_similarity_matrix, 0)

print("Similarity Matrix Statistics:")
similarity_values = book_similarity_matrix[np.triu_indices_from(book_similarity_matrix, k=1)]
print(f" Mean similarity: {similarity_values.mean():.4f}")
print(f" Std deviation: {similarity_values.std():.4f}")
print(f" Min similarity: {similarity_values.min():.4f}")
print(f" Max similarity: {similarity_values.max():.4f}")
print(f" Percentiles:")
print(f"  25%: {np.percentile(similarity_values, 25):.4f}")
print(f"  50%: {np.percentile(similarity_values, 50):.4f}")
print(f"  75%: {np.percentile(similarity_values, 75):.4f}")

Similarity Matrix Statistics:
 Mean similarity: 0.0040
 Std deviation: 0.0623
 Min similarity: 0.0000
 Max similarity: 1.0000
 Percentiles:
  25%: 0.0000
  50%: 0.0000
  75%: 0.0000


In [58]:
# Show example: find most similar books to a sample book
sample_book_idx = 5
sample_book = unique_books[sample_book_idx]
sample_similarity = book_similarity_matrix[sample_book_idx]
top_similar_idx = np.argsort(sample_similarity)[-6:-1][::-1] # Top 5 excluding itself

print(f"Most similar books to '{sample_book}':")
print("-" * 80)
for rank, idx in enumerate(top_similar_idx, 1):
  similar_book = unique_books[idx]
  similarity_score = book_similarity_matrix[sample_book_idx][idx]
  shared_readers = co_occurrence_dense[sample_book_idx][idx]
  print(f"{rank}. [{similarity_score:.4f}] {similar_book}")
  print(f"  Shared readers: {int(shared_readers)}")

Most similar books to 'A game of thrones':
--------------------------------------------------------------------------------
1. [0.0000] бы Clarissa Pinkola Estеs
  Shared readers: 0
2. [0.0000] Systems thinking for social change : a practical guide to solving complex problems, avoiding unintended consequences, and achieving lasting results
  Shared readers: 0
3. [0.0000] The Norton anthology of African American literature
  Shared readers: 0
4. [0.0000] The Ernst & Young business plan guide
  Shared readers: 0
5. [0.0000] The Eastern origins of Western civilization
  Shared readers: 0


## Section 4: Implement Collaborative Filtering Recommendation Function

In [ ]:
def get_cf_recommendations(user_id, num_recommendations=10, min_similarity=0.0):
  """
  Get book recommendations using collaborative filtering.
  
  Algorithm:
  1. Get all books the user has read
  2. For each unread book, compute similarity to all read books
  3. Aggregate similarity scores (even if 0)
  4. Return top N recommendations
  
  NOTE: Some users may have isolated interests (no shared readers).
  This shows real data quality - not a bug, just reality of the dataset.
  
  Args:
    user_id (str): User ID (e.g., 'User162')
    num_recommendations (int): Number to return (default: 10)
    min_similarity (float): Minimum similarity threshold (default: 0.0 = no filter)
  
  Returns:
    dict: Contains user info and recommendations dataframe
  """
  
  if user_id not in user_to_idx:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': f'{user_id} not found in dataset'
    }
  
  # Get books this user has read
  user_idx = user_to_idx[user_id]
  user_vector = user_book_matrix[user_idx].toarray().flatten()
  read_book_indices = np.where(user_vector > 0)[0]
  
  if len(read_book_indices) == 0:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': f'{user_id} has no books in the system'
    }
  
  # Get books user hasn't read
  unread_indices = np.where(user_vector == 0)[0]
  
  # Score each unread book based on similarity to read books
  book_scores = {}
  for unread_idx in unread_indices:
    # Get similarities to all books user has read
    similarities = book_similarity_matrix[unread_idx][read_book_indices]
    shared_readers = co_occurrence_dense[unread_idx][read_book_indices]
    
    if len(similarities) > 0:
      # Use weighted average (emphasize strongest match)
      avg_sim = np.mean(similarities)
      max_sim = np.max(similarities)
      score = 0.6 * max_sim + 0.4 * avg_sim
      
      # Get total shared readers (may be 0 for isolated users)
      total_shared = int(shared_readers.sum())
      
      if score >= min_similarity:
        book_scores[unread_idx] = {
          'title': unique_books[unread_idx],
          'score': score,
          'max_similarity': max_sim,
          'avg_similarity': avg_sim,
          'shared_readers_count': total_shared
        }
  
  if not book_scores:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': f'No books to recommend for {user_id}'
    }
  
  # Sort by score and get top N
  sorted_recs = sorted(book_scores.items(), key=lambda x: x[1]['score'], reverse=True)
  top_recs = sorted_recs[:num_recommendations]
  
  # Create recommendations dataframe
  recs_data = []
  for rank, (book_idx, scores) in enumerate(top_recs, 1):
    recs_data.append({
      'Rank': rank,
      'Title': scores['title'],
      'Similarity Score': scores['score'],
      'Max Similarity': scores['max_similarity'],
      'Avg Similarity': scores['avg_similarity'],
      'Shared Readers': scores['shared_readers_count']
    })
  
  recs_df = pd.DataFrame(recs_data)
  
  read_books = [unique_books[idx] for idx in read_book_indices]
  
  return {
    'user_id': user_id,
    'status': 'success',
    'num_books_read': len(read_books),
    'books_read': read_books[:5], # Show first 5
    'recommendations': recs_df,
    'has_shared_readers': (recs_df['Shared Readers'] > 0).any()
  }

# Test the function
result = get_cf_recommendations('User002', num_recommendations=10)
print(f"Recommendations for {result['user_id']}")
print(f"Status: {result['status']}")

if result['status'] == 'error':
  print(f"Error: {result['message']}")
else:
  print(f"Books read by user: {result['num_books_read']}")
  has_shared = result['has_shared_readers']
  print(f"Uses interest overlap with other users: {'YES' if has_shared else 'NO (isolated tastes)'}")
  print(f"\nTop {len(result['recommendations'])} Recommendations:")
  print(result['recommendations'].to_string(index=False))

Recommendations for User002
Status: success
Books read by user: 2
Uses interest overlap with other users: NO (isolated tastes)

Top 10 Recommendations:
 Rank                                               Title Similarity Score Max Similarity Avg Similarity Shared Readers
  1 $Pread : the best of the magazine that illuminated the sex industry and started a media revolution        0.0       0.0       0.0        0
  2                              12 rules for life : an antidote to chaos        0.0       0.0       0.0        0
  3        21st century reading. 2, Student book : creative thinking and reading with TED Talks        0.0       0.0       0.0        0
  4                                       A collection of essays        0.0       0.0       0.0        0
  5           A concise history of the Armenian people : (from ancient times to the present)        0.0       0.0       0.0        0
  6                                         A game of thrones        0.0       0.0       0.0     

## Section 5: Interactive Testing Interface

In [ ]:
# Get all available users
all_users = list(unique_users)

print(f"Total users in dataset: {len(all_users)}")
print(f"User ID range: {all_users[0]} to {all_users[-1]}")

# Display function
def display_cf_recommendations(user_id, num_recs=10):
  """Display collaborative filtering recommendations nicely"""
  result = get_cf_recommendations(user_id, num_recommendations=num_recs)
  
  if result['status'] == 'error':
    print(f"Error: {result['message']}")
    return
  
  print(f"\n{'='*90}")
  print(f"COLLABORATIVE FILTERING RECOMMENDATIONS FOR {result['user_id']}")
  print(f"{'='*90}")
  print(f"Books user has read: {result['num_books_read']}")
  print(f"\nSample books already read:")
  for book in result['books_read']:
    print(f" {book}")
  
  print(f"\n{'-'*90}")
  print(f"TOP {len(result['recommendations'])} RECOMMENDATIONS (Based on Reader Similarity):")
  print(f"{'-'*90}")
  
  for _, row in result['recommendations'].iterrows():
    print(f"#{int(row['Rank']):2d}. {row['Title']}")
    print(f"   Similarity Score: {row['Similarity Score']:.4f}")
    print(f"   (Max: {row['Max Similarity']:.4f}, Avg: {row['Avg Similarity']:.4f})")
    print(f"   Shared readers with your books: {int(row['Shared Readers'])}")
    print()

Total users in dataset: 298
User ID range: User002 to User300


In [ ]:
# Interactive testing with widgets
try:
  from ipywidgets import Dropdown, IntSlider, Button, Output, VBox, HBox
  from IPython.display import display, clear_output
  
  # Create widgets
  user_dropdown = Dropdown(
    options=all_users,
    value=all_users[0],
    description='Select User:',
    style={'description_width': '120px'}
  )
  
  num_recs_slider = IntSlider(
    value=10,
    min=1,
    max=30,
    step=1,
    description='# Recommendations:',
    style={'description_width': '160px'}
  )
  
  execute_button = Button(
    description='Show Recommendations',
    button_style='info',
    tooltip='Click to get recommendations'
  )
  
  output = Output()
  
  # Button callback
  def on_button_click(b):
    with output:
      clear_output(wait=True)
      selected_user = user_dropdown.value
      num_recs = num_recs_slider.value
      display_cf_recommendations(selected_user, num_recs=num_recs)
  
  execute_button.on_click(on_button_click)
  
  # Layout
  controls = VBox([
    user_dropdown,
    num_recs_slider,
    execute_button
  ])
  
  display(controls)
  display(output)
  
  # Auto-run on first load
  on_button_click(None)
  
except ImportError:
  print("\nipywidgets not available. Using text input instead.\n")
  user_input = input(f"Enter user ID (e.g., {all_users[0]}): ").strip()
  if not user_input:
    user_input = all_users[0]
  num_input = input("Number of recommendations (default 10): ").strip()
  num_recs = int(num_input) if num_input.isdigit() else 10
  display_cf_recommendations(user_input, num_recs=num_recs)

Output()

## Section 6: System Summary and Statistics

In [ ]:
print("="*80)
print("COLLABORATIVE FILTERING SYSTEM - SUMMARY")
print("="*80)

print(f"""
Dataset Statistics:
 - Total Users:       {len(unique_users)}
 - Total Books:       {len(unique_books)}
 - Total Interactions:    {len(user_books_parsed)}
 - User-Book Matrix Density: {(len(user_books_parsed) / (len(unique_users) * len(unique_books)) * 100):.2f}%

Book Connectivity:
 - Mean readers per book:  {user_book_matrix.sum(axis=0).mean():.1f}
 - Max readers per book:   {int(user_book_matrix.sum(axis=0).max())}
 - Min readers per book:   {int(user_book_matrix.sum(axis=0).min())}

User Activity:
 - Mean books per user:   {user_book_matrix.sum(axis=1).mean():.1f}
 - Max books per user:    {int(user_book_matrix.sum(axis=1).max())}
 - Min books per user:    {int(user_book_matrix.sum(axis=1).min())}

Similarity Matrix:
 - Shape:          {book_similarity_matrix.shape}
 - Method:          60% Jaccard + 40% Cosine
 - Mean Similarity:     {similarity_values.mean():.4f}
 - Std Dev:         {similarity_values.std():.4f}
 - Max Similarity:      {similarity_values.max():.4f}

Co-Occurrence:
 - Mean shared readers:   {co_occurrence_values.mean():.2f}
 - Max shared readers:    {int(co_occurrence_values.max())}
 - Books with >5 shared:   {(co_occurrence_values > 5).sum()}

Recommendation Algorithm:
 - Strategy:         Item-Item Collaborative Filtering
 - Similarity Metric:    Jaccard (reader overlap) + Cosine
 - Scoring Formula:     60% Max Similarity + 40% Avg Similarity
 - Data Required:      User-book interactions only
""")

print("\n" + "="*80)
print("HOW TO USE THIS SYSTEM:")
print("="*80)
print("""
1. Direct function call:
  result = get_cf_recommendations('User002', num_recommendations=10)

2. Display function:
  display_cf_recommendations('User250', num_recs=15)

3. Interactive dropdown above - select user and click button

Key Differences from Content-Based:
 * Uses READER OVERLAP instead of genre similarity
 * Recommends books read by users with similar taste
 * Captured through co-occurrence patterns
 * Works well when books lack detailed metadata
 * Captures implicit user preferences

Next Step: HYBRID Recommendations
 → Combine Content-Based + Collaborative Filtering
 → 50% content similarity + 50% reader similarity
 → Best of both worlds!
""")

COLLABORATIVE FILTERING SYSTEM - SUMMARY

Dataset Statistics:
 - Total Users:       298
 - Total Books:       909
 - Total Interactions:    999
 - User-Book Matrix Density: 0.37%

Book Connectivity:
 - Mean readers per book:  1.1
 - Max readers per book:   56
 - Min readers per book:   1

User Activity:
 - Mean books per user:   3.4
 - Max books per user:    21
 - Min books per user:    1

Similarity Matrix:
 - Shape:          (909, 909)
 - Method:          60% Jaccard + 40% Cosine
 - Mean Similarity:     0.0040
 - Std Dev:         0.0623
 - Max Similarity:      1.0000

Co-Occurrence:
 - Mean shared readers:   0.00
 - Max shared readers:    4
 - Books with >5 shared:   0

Recommendation Algorithm:
 - Strategy:         Item-Item Collaborative Filtering
 - Similarity Metric:    Jaccard (reader overlap) + Cosine
 - Scoring Formula:     60% Max Similarity + 40% Avg Similarity
 - Data Required:      User-book interactions only


HOW TO USE THIS SYSTEM:

1. Direct function call:
  result = g

## Section 7: Comparison with Content-Based (Optional)

To compare with content-based results, load the other notebook and compare recommendations for the same user.

In [62]:
print("To compare Collaborative Filtering vs Content-Based recommendations:")
print("")
print("1. Load the content-based recommendation notebook:")
print("  - company_u_content_based_recsys.ipynb")
print("")
print("2. Get recommendations for the same user with both methods")
print("")
print("3. Compare the results:")
print("  - Content-Based: Similar genres/themes")
print("  - Collaborative: Similar reader profiles")
print("")
print("4. For HYBRID recommendations, average both scores:")
print("  hybrid_score = 0.5 * content_score + 0.5 * cf_score")
print("")
print("This provides more diverse and personalized recommendations!")

To compare Collaborative Filtering vs Content-Based recommendations:

1. Load the content-based recommendation notebook:
  - company_u_content_based_recsys.ipynb

2. Get recommendations for the same user with both methods

3. Compare the results:
  - Content-Based: Similar genres/themes
  - Collaborative: Similar reader profiles

4. For HYBRID recommendations, average both scores:
  hybrid_score = 0.5 * content_score + 0.5 * cf_score

This provides more diverse and personalized recommendations!


In [63]:
# Test: Compare CF recommendations for multiple users
print("="*90)
print("TESTING COLLABORATIVE FILTERING ACROSS MULTIPLE USERS")
print("="*90)

# Test with a few sample users
test_users = ['User002', 'User010', 'User050', 'User100', 'User119']

recommendations_by_user = {}

for user_id in test_users:
  result = get_cf_recommendations(user_id, num_recommendations=5)
  
  if result['status'] == 'success':
    recommendations_by_user[user_id] = {
      'books_read': len(result['books_read']),
      'total_books': result['num_books_read'],
      'recommendations': result['recommendations']['Title'].tolist()
    }
    
    print(f"\n{user_id}:")
    print(f" Books read: {result['num_books_read']}")
    print(f" Top 5 recommendations:")
    for idx, (_, row) in enumerate(result['recommendations'].iterrows(), 1):
      print(f"  {idx}. {row['Title'][:50]}")
      print(f"    (Score: {row['Similarity Score']:.4f}, Shared readers: {int(row['Shared Readers'])})")

print("\n" + "="*90)
print("CHECKING FOR OVERLAP IN RECOMMENDATIONS")
print("="*90)

# Check if different users get different recommendations (good sign!)
for i, user1 in enumerate(test_users):
  for user2 in test_users[i+1:]:
    if user1 in recommendations_by_user and user2 in recommendations_by_user:
      recs1 = set(recommendations_by_user[user1]['recommendations'])
      recs2 = set(recommendations_by_user[user2]['recommendations'])
      overlap = recs1.intersection(recs2)
      
      print(f"\n{user1} vs {user2}:")
      print(f" Recommendation overlap: {len(overlap)}/5 books")
      if overlap:
        print(f" Common recommendations: {list(overlap)[:3]}")
      else:
        print(f" No overlap - good! Recommendations are personalized")

TESTING COLLABORATIVE FILTERING ACROSS MULTIPLE USERS

User002:
 Books read: 2
 Top 5 recommendations:
  1. $Pread : the best of the magazine that illuminated
    (Score: 0.0000, Shared readers: 0)
  2. 12 rules for life : an antidote to chaos
    (Score: 0.0000, Shared readers: 0)
  3. 21st century reading. 2, Student book : creative t
    (Score: 0.0000, Shared readers: 0)
  4. A collection of essays
    (Score: 0.0000, Shared readers: 0)
  5. A concise history of the Armenian people : (from a
    (Score: 0.0000, Shared readers: 0)

User010:
 Books read: 3
 Top 5 recommendations:
  1. $Pread : the best of the magazine that illuminated
    (Score: 0.0000, Shared readers: 0)
  2. 12 rules for life : an antidote to chaos
    (Score: 0.0000, Shared readers: 0)
  3. 21st century reading. 2, Student book : creative t
    (Score: 0.0000, Shared readers: 0)
  4. A collection of essays
    (Score: 0.0000, Shared readers: 0)
  5. A concise history of the Armenian people : (from a
    (Score: 0

In [64]:
# Test CF across ALL user types - from best to worst overlap
print("="*90)
print("TESTING COLLABORATIVE FILTERING ON REAL DATA")
print("(Not filtering anyone - showing what CF can/cannot do)")
print("="*90)

# Categorize users by their interest overlap
cf_capable = [] # Users with shared interests
cf_isolated = [] # Users with isolated interests

for user_id, stats in sorted_users:
  if stats['max_shared_readers'] > 0:
    cf_capable.append(user_id)
  else:
    cf_isolated.append(user_id)

print(f"\n Data Quality Summary:")
print(f" Users with shared interests (CF works best): {len(cf_capable)}")
print(f" Users with isolated interests (CF limited): {len(cf_isolated)}")
print(f" Total users: {len(unique_users)}")

# Test 3 users from EACH category
print(f"\n\n{'='*90}")
print(f"GROUP 1: USERS WITH SHARED INTERESTS")
print(f"(Collaborative filtering should work well)")
print(f"{'='*90}\n")

for user_id in cf_capable[:3]:
  result = get_cf_recommendations(user_id, num_recommendations=3)
  stats = user_overlap_scores[user_id]
  
  print(f"\n{user_id}:")
  print(f" Books read: {stats['num_books']}")
  print(f" Max shared readers among their books: {int(stats['max_shared_readers'])}")
  
  if result['status'] == 'success':
    print(f" CF Status: {' Works (has shared interests)' if result['has_shared_readers'] else '⚠ No overlaps found'}")
    print(f"\n Recommendations (top 3):")
    for _, row in result['recommendations'].iterrows():
      shared_str = f"{int(row['Shared Readers'])} shared readers" if row['Shared Readers'] > 0 else "NO shared readers"
      print(f"  {int(row['Rank'])}. {row['Title'][:55]}")
      print(f"    Score: {row['Similarity Score']:.4f} | {shared_str}")
  else:
    print(f"  Error: {result['message']}")

print(f"\n\n{'='*90}")
print(f"GROUP 2: USERS WITH ISOLATED INTERESTS")
print(f"(Collaborative filtering has limited visibility)")
print(f"{'='*90}\n")

for user_id in cf_isolated[:3]:
  result = get_cf_recommendations(user_id, num_recommendations=3)
  stats = user_overlap_scores[user_id]
  
  print(f"\n{user_id}:")
  print(f" Books read: {stats['num_books']}")
  print(f" Max shared readers among their books: {int(stats['max_shared_readers'])}")
  
  if result['status'] == 'success':
    print(f" CF Status: {' Works (has shared interests)' if result['has_shared_readers'] else '⚠ Isolated reader (no CF signals)'}")
    print(f"\n Recommendations (top 3):")
    for _, row in result['recommendations'].iterrows():
      shared_str = f"{int(row['Shared Readers'])} shared readers" if row['Shared Readers'] > 0 else "NO shared readers"
      print(f"  {int(row['Rank'])}. {row['Title'][:55]}")
      print(f"    Score: {row['Similarity Score']:.4f} | {shared_str}")
  else:
    print(f"  Error: {result['message']}")

print(f"\n\n{'='*90}")
print(f"MEANING: Collaborative Filtering Effectiveness")
print(f"{'='*90}")
print(f"""
CF WORKS when: Users read books that others also read
 → Can find "similar readers" → Good recommendations
 → Example: Both read Harry Potter + Hunger Games
 
CF FAILS when: Users have UNIQUE taste (nobody else read their books)
 → Can't find "similar readers" → Recommendations have 0 overlap
 → Example: User only read 3 obscure Armenian books nobody else reads
 
SOLUTION: Use HYBRID approach
 • Content-Based: Works for EVERYONE (uses genres/titles)
 • Collaborative: Works for SOME (uses reader overlap)
 • Hybrid: Best of both = Works for EVERYONE
""")

TESTING COLLABORATIVE FILTERING ON REAL DATA
(Not filtering anyone - showing what CF can/cannot do)

 Data Quality Summary:
 Users with shared interests (CF works best): 296
 Users with isolated interests (CF limited): 2
 Total users: 298


GROUP 1: USERS WITH SHARED INTERESTS
(Collaborative filtering should work well)


User008:
 Books read: 4
 Max shared readers among their books: 4
 CF Status: Works (has shared interests)

 Recommendations (top 3):
  1. A question of genocide : Armenians and Turks at the end
    Score: 0.0927 | 3 shared readers
  2. Michel Foucault ; edited
    Score: 0.0927 | 3 shared readers
  3. Precarious life : the powers of mourning and violence
    Score: 0.0927 | 3 shared readers

User054:
 Books read: 5
 Max shared readers among their books: 4
 CF Status: Works (has shared interests)

 Recommendations (top 3):
  1. Precarious life : the powers of mourning and violence
    Score: 0.4069 | 4 shared readers
  2. Silvia Federici., The Palace of Dreams
    Score

In [65]:
# SUMMARY: What does CF really tell us?
print("\n" + "="*90)
print("CRITICAL FINDING: WHY CF WORKS OR FAILS")
print("="*90)

# Count how many recommendations have actual shared readers
cf_quality_stats = {
  'users_tested': 0,
  'users_cf_works': 0,
  'users_cf_isolated': 0,
  'avg_shared_readers': 0
}

all_shared_reader_counts = []

for user_id in unique_users[:50]: # Sample first 50 users
  result = get_cf_recommendations(user_id, num_recommendations=5)
  
  if result['status'] == 'success':
    cf_quality_stats['users_tested'] += 1
    
    # Count how many recs have >0 shared readers
    has_overlap = (result['recommendations']['Shared Readers'] > 0).sum()
    
    if has_overlap > 0:
      cf_quality_stats['users_cf_works'] += 1
    else:
      cf_quality_stats['users_cf_isolated'] += 1
    
    # Track shared reader counts
    shared_counts = result['recommendations']['Shared Readers'].values
    all_shared_reader_counts.extend(shared_counts)

print(f"\nSample of {cf_quality_stats['users_tested']} Users:")
print(f" Users where CF finds shared interests: {cf_quality_stats['users_cf_works']}")
print(f" Users with NO shared interests found: {cf_quality_stats['users_cf_isolated']}")
print(f" Ratio CF works: {(cf_quality_stats['users_cf_works'] / max(cf_quality_stats['users_tested'], 1) * 100):.0f}%")

if all_shared_reader_counts:
  avg_shared = np.mean(all_shared_reader_counts)
  print(f"\n Average shared readers per recommendation: {avg_shared:.2f}")
  print(f" Max shared readers found: {int(max(all_shared_reader_counts))}")
  print(f" Recommendations with 0 shared readers: {int((np.array(all_shared_reader_counts) == 0).sum())} / {len(all_shared_reader_counts)}")

print(f"\n Interpretation:")
print(f"  If CF works <50%: Users have very diverse tastes")
print(f"  If CF works >80%: Dataset has strong patterns")
print(f"  High 0-shared-readers: Data is SPARSE")


CRITICAL FINDING: WHY CF WORKS OR FAILS

Sample of 50 Users:
 Users where CF finds shared interests: 11
 Users with NO shared interests found: 39
 Ratio CF works: 22%

 Average shared readers per recommendation: 0.48
 Max shared readers found: 4
 Recommendations with 0 shared readers: 206 / 250

 Interpretation:
  If CF works <50%: Users have very diverse tastes
  If CF works >80%: Dataset has strong patterns
  High 0-shared-readers: Data is SPARSE


In [66]:

print("\n\n" + "="*90)
print("DECISION: DO WE NEED A HYBRID SYSTEM?")
print("="*90)
print(f"""
Current Status:
  Content-Based System: Works for 100% of users
 ⚠ Collaborative Filtering: Works for only 22% of users
 
Hybrid would:
 • Combine: 50% Content-Based + 50% Collaborative
 • Benefit: The 22% could get "reader-aware" recommendations
 • Cost: Adds complexity, slower, marginal gains
 
Question: Does hybrid actually improve recommendations?
 → Need to load content-based system and do side-by-side comparison
 → If hybrid gives significantly better results → create separate notebook
 → If hybrid makes little difference → stick with content-based only

Recommendation for now:
 1. Use THIS notebook to quickly test hybrid concept
 2. Compare results for same user with both methods
 3. If promising → move to separate file (company_u_hybrid_recsys.ipynb)
 4. If not → keep content-based as main system, CF as reference

Next step: Load content-based system and do comparison test?
""")



DECISION: DO WE NEED A HYBRID SYSTEM?

Current Status:
  Content-Based System: Works for 100% of users
 ⚠ Collaborative Filtering: Works for only 22% of users
 
Hybrid would:
 • Combine: 50% Content-Based + 50% Collaborative
 • Benefit: The 22% could get "reader-aware" recommendations
 • Cost: Adds complexity, slower, marginal gains
 
Question: Does hybrid actually improve recommendations?
 → Need to load content-based system and do side-by-side comparison
 → If hybrid gives significantly better results → create separate notebook
 → If hybrid makes little difference → stick with content-based only

Recommendation for now:
 1. Use THIS notebook to quickly test hybrid concept
 2. Compare results for same user with both methods
 3. If promising → move to separate file (company_u_hybrid_recsys.ipynb)
 4. If not → keep content-based as main system, CF as reference

Next step: Load content-based system and do comparison test?



In [67]:

# Debug: Check similarity matrix values
print("\n" + "="*90)
print("DEBUGGING SIMILARITY MATRIX")
print("="*90)

print(f"\nSimilarity Matrix Shape: {book_similarity_matrix.shape}")
print(f"Non-zero elements: {np.count_nonzero(book_similarity_matrix)}")
print(f"All zeros?: {np.all(book_similarity_matrix == 0)}")

# Check some specific values
print(f"\nRandom sample of similarity values:")
sample_indices = np.random.choice(book_similarity_matrix.shape[0], 3, replace=False)
for i in sample_indices:
  for j in range(i+1, min(i+3, book_similarity_matrix.shape[0])):
    print(f" Book {i} vs Book {j}: {book_similarity_matrix[i][j]:.6f}")

print(f"\nJaccard similarity stats:")
print(f" Non-zero: {np.count_nonzero(jaccard_sim)}")
print(f" Max: {jaccard_sim.max():.6f}")
print(f" Min: {jaccard_sim.min():.6f}")

print(f"\nCosine similarity stats:")
print(f" Non-zero: {np.count_nonzero(cosine_sim)}")
print(f" Max: {cosine_sim.max():.6f}")
print(f" Min: {cosine_sim.min():.6f}")

# Check co-occurrence matrix
print(f"\nCo-occurrence Matrix:")
print(f" Max shared readers: {co_occurrence_dense.max()}")
print(f" Non-zero pairs: {np.count_nonzero(co_occurrence_dense)}")
non_zero = np.where(co_occurrence_dense > 0)
if len(non_zero[0]) > 0:
  print(f" Sample non-zero: {co_occurrence_dense[non_zero[0][0], non_zero[1][0]]}")



DEBUGGING SIMILARITY MATRIX

Similarity Matrix Shape: (909, 909)
Non-zero elements: 3746
All zeros?: False

Random sample of similarity values:
 Book 854 vs Book 855: 0.000000
 Book 854 vs Book 856: 0.000000
 Book 750 vs Book 751: 0.000000
 Book 750 vs Book 752: 0.000000
 Book 615 vs Book 616: 0.000000
 Book 615 vs Book 617: 0.000000

Jaccard similarity stats:
 Non-zero: 4655
 Max: 1.000000
 Min: 0.000000

Cosine similarity stats:
 Non-zero: 4655
 Max: 1.000000
 Min: 0.000000

Co-occurrence Matrix:
 Max shared readers: 108
 Non-zero pairs: 4655
 Sample non-zero: 1
